<a href="https://colab.research.google.com/github/hidarihizi/Trainee/blob/main/%E3%82%A4%E3%83%B3%E3%82%BF%E3%83%BC%E3%83%B3_%E5%AE%8C%E6%88%90%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# -----------------------------------------------------
# 1. データ読み込みと前処理
# -----------------------------------------------------
try:
    df = pd.read_csv('stock_price.csv')
except FileNotFoundError:
    print("stock_price.csv' が見つかりません。")
    raise

# 日付変換とソート
df['日付'] = pd.to_datetime(df['日付'])
df = df.sort_values('日付').reset_index(drop=True)

# 出来高のクリーニング関数
def clean_volume_robust(x):
    if isinstance(x, (int, float)): return x
    x = str(x).upper()
    if 'M' in x: return float(x.replace('M', '')) * 1_000_000
    elif 'K' in x: return float(x.replace('K', '')) * 1_000
    elif 'B' in x: return float(x.replace('B', '')) * 1_000_000_000
    return float(x)

df['出来高'] = df['出来高'].apply(clean_volume_robust)
df['Day'] = df['日付'].dt.day
df['Month_ID'] = df['日付'].dt.to_period('M')

# -----------------------------------------------------
# 2. 特徴量エンジニアリング
# -----------------------------------------------------
# (1) 移動平均乖離率 (MA_Gap)
df['MA25'] = df['終値'].rolling(window=25).mean()
df['MA_Gap'] = (df['終値'] - df['MA25']) / df['MA25'] * 100

# (2) RSI (相対力指数)
delta = df['終値'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# (3) 出来高倍率 (Volume_Ratio)
df['Vol_MA15'] = df['出来高'].rolling(window=15).mean()
df['Volume_Ratio'] = df['出来高'] / df['Vol_MA15']

# 欠損値除去
df = df.dropna().reset_index(drop=True)

# -----------------------------------------------------
# 3. 正解ラベルの作成と分割
# -----------------------------------------------------
def set_target(x):
    criterion = x['終値'].quantile(0.3)
    return (x['終値'] <= criterion).astype(int)

df['Target'] = df.groupby('Month_ID', group_keys=False).apply(set_target)

# データ分割（2023年を境界）
test_start_date = '2023-01-01'
train = df[df['日付'] < test_start_date].copy()
test = df[df['日付'] >= test_start_date].copy()

features = ['RSI', 'MA_Gap', 'Volume_Ratio', 'Day']
X_train = train[features]
y_train = train['Target']
X_test = test[features]
y_test = test['Target']

# -----------------------------------------------------
# 4. モデル学習
# -----------------------------------------------------
# ★過学習を抑えるため、少しパラメータをマイルドに戻しました
final_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.1,
    min_child_samples=10,
    verbose=-1
)

final_model.fit(X_train, y_train)

# -----------------------------------------------------
# 5. 学習データ内での閾値決定 (Fair Validation)
# -----------------------------------------------------
# まず、学習データ自身に対する予測確率を出す
train_probs = final_model.predict_proba(X_train)[:, 1]
train['Probability'] = train_probs

best_k = 0.5
max_train_profit = -100

# 学習期間（過去）の中で、一番成績が良かった閾値を探す
for k in np.arange(0.50, 0.90, 0.01):
    temp_action = (train['Probability'] >= k).astype(int)
    temp_buys = train[temp_action == 1]

    if len(temp_buys) > 10: # ある程度のサンプル数が必要
        # 簡易利益計算（学習期間内）
        p_sum = 0
        v_months = temp_buys['Month_ID'].unique()
        for m in v_months:
            ai_p = temp_buys[temp_buys['Month_ID'] == m]['終値'].mean()
            # 山田さん（月末と比較）※簡易計算
            sato_data = train[(train['Month_ID'] == m) & (train['Day'] >= 25)]
            if len(sato_data) > 0: sato_p = sato_data.iloc[0]['終値']
            else: sato_p = train[train['Month_ID'] == m].iloc[-1]['終値']
            p_sum += (sato_p - ai_p)

        avg_p = p_sum / len(v_months)

        if avg_p > max_train_profit:
            max_train_profit = avg_p
            best_k = k

print(f"決定された運用閾値: {best_k:.2f} (過去データより算出)")

# -----------------------------------------------------
# 6. テストデータでの本番評価
# -----------------------------------------------------
test_probs = final_model.predict_proba(X_test)[:, 1]
test['Probability'] = test_probs
test['My_Action'] = (test['Probability'] >= best_k).astype(int)

# 利益計算
my_buys = test[test['My_Action'] == 1]

if len(my_buys) == 0:
    final_profit = 0
else:
    profit_sum = 0
    valid_months = my_buys['Month_ID'].unique()
    for m in valid_months:
        ai_p = my_buys[my_buys['Month_ID'] == m]['終値'].mean()
        sato_data = test[(test['Month_ID'] == m) & (test['Day'] >= 25)]
        if len(sato_data) > 0: sato_p = sato_data.iloc[0]['終値']
        else: sato_p = test[test['Month_ID'] == m].iloc[-1]['終値']
        profit_sum += (sato_p - ai_p)
    final_profit = profit_sum / len(valid_months)

# 結果出力
yamada_avg_p = 96.13
improvement_rate = (final_profit / yamada_avg_p) * 100

print("-" * 30)
print(f"改善幅: +{final_profit:.2f} 円")
print(f"改善率: {improvement_rate:.2f} %")
print("-" * 30)

if improvement_rate >= 1.5:
    print(">> 買い時です（1.5%以上の効果あり）")
elif improvement_rate > 0:
    print(">> プラス収支ですが、目標(1.5%)には届きませんでした")
else:
    print(">> 効果なし（マイナスまたは取引なし）")

# -----------------------------------------------------
# 7. 実戦配備：本日のシグナル判定
# -----------------------------------------------------
latest_row = test.iloc[[-1]].copy()
target_date = latest_row['日付'].dt.strftime('%Y年%m月%d日').values[0]
target_prob = latest_row['Probability'].values[0]

print("\n" + "="*40)
print(f"{target_date} のAI判定")
print("="*40)

if target_prob >= best_k:
    print(f"【買いシグナル点灯】")
    print(f"確信度: {target_prob*100:.1f}% (合格ライン {best_k*100:.1f}%)")
    print(f"結論: {target_date} は『買い時』です。")
else:
    print(f"【様子見モード】")
    print(f"確信度: {target_prob*100:.1f}% (合格ライン {best_k*100:.1f}%)")
    print(f"結論: まだ慌てる時間ではありません。")

決定された運用閾値: 0.50 (過去データより算出)
------------------------------
改善幅: +2.00 円
改善率: 2.08 %
------------------------------
>> 買い時です（1.5%以上の効果あり）

2025年12月30日 のAI判定
【様子見モード】
確信度: 1.6% (合格ライン 50.0%)
結論: まだ慌てる時間ではありません。
